# Notebook 04: Supervised Salary Prediction (Regression)
**Project:** JobLens Vietnam - Labor Market Analysis & Skill Trends  
**Course:** Introduction to Data Science  

### Objective:
1. Predict monthly salary in Million VND (`salary_mean_vnd`) using multi-modal features.
2. Combine 384-dimensional SBERT text embeddings with structured features:
   - `job_role_label`
   - `exp_years` (numerical experience)
   - `norm_location` (regional location)
   - `norm_position` (job position hierarchy)
   - `norm_job_type` (employment type)
   - `year_clean` (mandatory year feature)
3. Evaluate 6 regression algorithms:
   - Dummy Regressor (Median baseline)
   - Linear Regression
   - Ridge Regression
   - Random Forest Regressor
   - Extra Trees Regressor
   - XGBoost Regressor
4. Conduct required **Ablation Study**:
   - E1: Structured features without Year
   - E2: Structured features with Year
   - E3: SBERT + Structured features without Year
   - E4: SBERT + Structured features with Year
5. Analyze Actual vs Predicted salary, residuals, and feature importances.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.models.regression import run_regression_experiments

## 1. Load Regression Dataset and Embeddings
Filter records with valid numerical monthly salaries and retrieve corresponding SBERT description embeddings.

In [ ]:
df_reg = pd.read_csv(config.PROCESSED_DATA_DIR / "regression_data.csv")
sbert_all = np.load(config.EMBEDDINGS_DIR / "sbert_embeddings.npy")
cleaned_df = pd.read_csv(config.PROCESSED_DATA_DIR / "cleaned_jobs.csv")

id_to_idx = {uid: idx for idx, uid in enumerate(cleaned_df['unique_job_id'])}
reg_indices = [id_to_idx[uid] for uid in df_reg['unique_job_id']]
sbert_reg = sbert_all[reg_indices]

print(f"Regression samples: {len(df_reg)}")
print("Salary target summary (Million VND/month):")
print((df_reg['salary_mean_vnd'] / 1_000_000).describe().round(2))

## 2. Model Training and Ablation Study
We train all 6 regressors on configuration E4 (full multi-modal feature set) and evaluate ablation configurations E1 through E4.

In [ ]:
reg_results, ablation_results, best_model, metadata = run_regression_experiments(df_reg, sbert_reg)

print("=== Full Model Comparison (Configuration E4) ===")
display(reg_results) if 'display' in dir() else print(reg_results)

print("\n=== Ablation Study Summary (E1, E2, E3, E4) ===")
display(ablation_results) if 'display' in dir() else print(ablation_results)

## 3. Regression Diagnostic Plots
Examine the Actual vs Predicted scatter plot, residual error distributions, feature importances, and ablation comparison.

In [ ]:
from IPython.display import Image, display as ipy_display

for plot_name in ['actual_vs_predicted.png', 'residual_plot.png', 'feature_importance.png', 'ablation_comparison.png']:
    p = config.REGRESSION_OUTPUTS / 'plots' / plot_name
    if p.exists():
        print(f"\n--- {plot_name} ---")
        ipy_display(Image(filename=str(p)))